# S10 — EA1 | Carga y creación de dataset
**Estudiante:** José David Osorio Gallego  
**Actividad:** Obtener datos de Hugging Face, transformarlos con pandas y documentar el resultado.

Elegí el dataset [Iris de Hugging Face](https://huggingface.co/datasets/scikit-learn/iris) porque permite entender fácilmente las columnas: son medidas de flores y su especie. En este cuaderno separo las responsabilidades: una clase descarga los datos, otra construye el resumen, y aquí hago las transformaciones.

## 1. Preparación y carga

La clase `HFDatasetLoader` se conecta a Hugging Face. Utilizo el split `train` y lo convierto a un DataFrame, sin modificar los datos originales.

In [1]:
import pandas as pd
from IPython.display import display

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

In [2]:
cargador = HFDatasetLoader("scikit-learn/iris", split="train")
df_original = cargador.cargar()

print(f"Datos originales: {df_original.shape[0]} filas y {df_original.shape[1]} columnas")
print("Columnas originales:", ", ".join(df_original.columns))
display(df_original.head())

Datos originales: 150 filas y 6 columnas
Columnas originales: Id, SepalLengthCm, SepalWidthCm, PetalLengthCm, PetalWidthCm, Species
 Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm     Species
  1            5.1           3.5            1.4           0.2 Iris-setosa
  2            4.9           3.0            1.4           0.2 Iris-setosa
  3            4.7           3.2            1.3           0.2 Iris-setosa
  4            4.6           3.1            1.5           0.2 Iris-setosa
  5            5.0           3.6            1.4           0.2 Iris-setosa


## 2. Transformaciones del DataFrame

La rúbrica solicita **al menos cuatro transformaciones distintas**. Realizo cinco, mostrando lo que ocurre después de cada una. No cuento `head()` como transformación.

In [3]:
# Transformación 1: eliminar un identificador que no representa una medición.
df_sin_id = df_original.drop(columns=["Id"])
print("1. Eliminar Id:", df_sin_id.shape)
display(df_sin_id.head(3))

1. Eliminar Id: (150, 5)
 SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm     Species
           5.1           3.5            1.4           0.2 Iris-setosa
           4.9           3.0            1.4           0.2 Iris-setosa
           4.7           3.2            1.3           0.2 Iris-setosa


In [4]:
# Transformación 2: dar nombres descriptivos en español.
df_renombrado = df_sin_id.rename(
    columns={
        "SepalLengthCm": "largo_sepalo_cm",
        "SepalWidthCm": "ancho_sepalo_cm",
        "PetalLengthCm": "largo_petalo_cm",
        "PetalWidthCm": "ancho_petalo_cm",
        "Species": "especie",
    }
)
print("2. Columnas renombradas:", df_renombrado.columns.tolist())
display(df_renombrado.head(3))

2. Columnas renombradas: ['largo_sepalo_cm', 'ancho_sepalo_cm', 'largo_petalo_cm', 'ancho_petalo_cm', 'especie']
 largo_sepalo_cm  ancho_sepalo_cm  largo_petalo_cm  ancho_petalo_cm     especie
             5.1              3.5              1.4              0.2 Iris-setosa
             4.9              3.0              1.4              0.2 Iris-setosa
             4.7              3.2              1.3              0.2 Iris-setosa


In [5]:
# Transformación 3: conservar flores cuyo pétalo mida más de 1.5 cm.
df_filtrado = df_renombrado.loc[
    df_renombrado["largo_petalo_cm"] > 1.5
].copy().reset_index(drop=True)
print(f"3. Filtrado: {len(df_original)} -> {len(df_filtrado)} registros")
display(df_filtrado.head(3))

3. Filtrado: 150 -> 113 registros
 largo_sepalo_cm  ancho_sepalo_cm  largo_petalo_cm  ancho_petalo_cm     especie
             5.4              3.9              1.7              0.4 Iris-setosa
             4.8              3.4              1.6              0.2 Iris-setosa
             5.7              3.8              1.7              0.3 Iris-setosa


In [6]:
# Transformación 4: calcular una medida aproximada del tamaño del pétalo.
# Largo x ancho es una aproximación rectangular, no el área real de la flor.
df_con_area = df_filtrado.copy()
df_con_area["area_petalo_aprox_cm2"] = (
    df_con_area["largo_petalo_cm"] * df_con_area["ancho_petalo_cm"]
).round(2)
print("4. Columna calculada: area_petalo_aprox_cm2")
display(df_con_area.head(3))

4. Columna calculada: area_petalo_aprox_cm2
 largo_sepalo_cm  ancho_sepalo_cm  largo_petalo_cm  ancho_petalo_cm     especie  area_petalo_aprox_cm2
             5.4              3.9              1.7              0.4 Iris-setosa                   0.68
             4.8              3.4              1.6              0.2 Iris-setosa                   0.32
             5.7              3.8              1.7              0.3 Iris-setosa                   0.51


In [7]:
# Transformación 5: ordenar de mayor a menor área aproximada.
df_final = df_con_area.sort_values(
    by="area_petalo_aprox_cm2", ascending=False
).reset_index(drop=True)

print("5. Ordenar de mayor a menor:", df_final.shape)
display(df_final.head(10))
print("\nCantidad por especie después del filtro:")
print(df_final["especie"].value_counts().to_string())

5. Ordenar de mayor a menor: (113, 6)
 largo_sepalo_cm  ancho_sepalo_cm  largo_petalo_cm  ancho_petalo_cm        especie  area_petalo_aprox_cm2
             7.7              2.6              6.9              2.3 Iris-virginica                  15.87
             7.2              3.6              6.1              2.5 Iris-virginica                  15.25
             6.3              3.3              6.0              2.5 Iris-virginica                  15.00
             7.7              3.8              6.7              2.2 Iris-virginica                  14.74
             6.7              3.3              5.7              2.5 Iris-virginica                  14.25
             7.7              3.0              6.1              2.3 Iris-virginica                  14.03
             7.6              3.0              6.6              2.1 Iris-virginica                  13.86
             6.8              3.2              5.9              2.3 Iris-virginica                  13.57
        

## 3. Archivo de resumen

La segunda clase recibe **`df_final`**, no el DataFrame original. Así garantizo que el archivo `resumen_iris_transformado.txt` describe las columnas y las filas que quedaron después del procesamiento.

In [8]:
resumidor = DatasetSummarizer(df_final)
ruta_resumen = resumidor.guardar_txt("resumen_iris_transformado.txt")
texto_resumen = ruta_resumen.read_text(encoding="utf-8")
print(f"Resumen guardado en: {ruta_resumen.resolve()}\n")
print(texto_resumen)

Resumen guardado en: /tmp/s10-ea1-aprende-python-run/S10-EA1-Carga-Creacion-Dataset/resumen_iris_transformado.txt

RESUMEN DEL DATASET IRIS TRANSFORMADO
Filas: 113
Columnas: 6

COLUMNAS Y TIPOS DE DATOS
----------------------------
- largo_sepalo_cm: float64
- ancho_sepalo_cm: float64
- largo_petalo_cm: float64
- ancho_petalo_cm: float64
- especie: str
- area_petalo_aprox_cm2: float64

VALORES NULOS POR COLUMNA
----------------------------
- largo_sepalo_cm: 0
- ancho_sepalo_cm: 0
- largo_petalo_cm: 0
- ancho_petalo_cm: 0
- especie: 0
- area_petalo_aprox_cm2: 0
Total de valores nulos: 0
Filas duplicadas: 1

ESTADÍSTICAS DESCRIPTIVAS (COLUMNAS NUMÉRICAS)
------------------------------------------------
       largo_sepalo_cm  ancho_sepalo_cm  largo_petalo_cm  ancho_petalo_cm  area_petalo_aprox_cm2
count           113.00           113.00           113.00           113.00                 113.00
mean              6.12             2.94             4.53             1.52                   7.5

## 4. Comprobación final

Estas verificaciones permiten detectar si por error se resume otro DataFrame o si alguna transformación no se aplicó.

In [9]:
assert isinstance(df_original, pd.DataFrame)
assert df_original.shape == (150, 6)
assert df_final.shape == (113, 6)
assert "Id" not in df_final.columns
assert "area_petalo_aprox_cm2" in df_final.columns
assert (df_final["largo_petalo_cm"] > 1.5).all()
assert df_final["area_petalo_aprox_cm2"].is_monotonic_decreasing
assert "Filas: 113" in texto_resumen
assert "Columnas: 6" in texto_resumen
assert "area_petalo_aprox_cm2" in texto_resumen

print("Comprobaciones correctas: carga, 5 transformaciones y TXT final.")

Comprobaciones correctas: carga, 5 transformaciones y TXT final.


## 5. ¿Qué aprendí?

El dataset comenzó con **150 filas y 6 columnas**. Al quitar el identificador, traducir las columnas, filtrar los pétalos de más de 1,5 cm, calcular una medida aproximada y ordenar el resultado, terminé con **113 filas y 6 columnas**. Quedaron 50 registros de *Iris-virginica*, 50 de *Iris-versicolor* y 13 de *Iris-setosa*.

El filtro cambia la proporción de especies, por lo que el conjunto final no tiene la misma distribución que el original. También encontré una fila duplicada exacta en el conjunto transformado: la reporto en el TXT, pero no la elimino porque la actividad solo requiere un procesamiento inicial. El área calculada es una aproximación rectangular para comparar observaciones, no una medición botánica exacta.

La principal ventaja de separar el trabajo en dos clases es que puedo volver a usar la carga y el resumen con otros datasets en las próximas actividades.